# Stochastic random walk kernels

## Imports

In [1]:
import numpy as np
import networkx as nx
import scipy.linalg as la
from scipy.sparse.linalg import cg, LinearOperator
import math
import random as rnd
import time
import scipy

## Some functions

In [2]:
def graph_generator(n, kind="er", seed=None):
    """
    er for Erdos-Renyi;
    ba for Barabasi-Albert;
    ws for Watts-Strogtz (small-world);
    sbm for Stochastic Block Model
    """
    
    if kind == "er":
        #Erdos-Renyi
        #p = 2.0/n gives us a moderately sparse graph:
        #E[deg] = (n - 1) * p = 2
        return nx.erdos_renyi_graph(n=n, p=float(2.0/n), seed=seed)
    
    if kind == "ba":
        #Barabasi-Albert (preferential attachment)
        # Each new node connects to m = max(1, n // 20) existing nodes.
        # This yields a scale-free graph with hubs.
        return nx.barabasi_albert_graph(n=n, m=max(1, n // 20), seed=seed)
    
    if kind == "ws":
        #Watts-Strogtz (small-world)
        # Start with a ring where each node connects to k neighbors, then rewire edges with p = 0.1.
        # This keeps high clustering while creating short average paths.
        k = int(max(2, (n // 10) | 1))
        p = float(0.1)
        return nx.watts_strogatz_graph(n=n, k=k, p=p, seed=seed)

    if kind == "sbm":
        # Stochastic Block Model with 2 groups.
        # Connect nodes within the same group with p_in = 0.15,
        # and across groups with p_out = 0.02 (weaker connections).
        sizes = [n//2, n - n//2]
        p_in, p_out = float(0.15), float(0.02)
        P = [
            [p_in,  p_out],
            [p_out, p_in]
        ]
        return nx.stochastic_block_model(sizes, P, seed=seed)
    
    raise ValueError(f"unknown kind: {kind}")

In [3]:
def graph_generator_labeled(n, kind="er", n_labels=3, seed=None):
    g = graph_generator(n, kind=kind, seed=seed)
    rng = np.random.default_rng(seed)
    for u, v in g.edges():
        g[u][v]["label"] = int(rng.integers(0, n_labels))
    return g

In [4]:
def normalized_adj_matrix(graph):
    A = nx.to_numpy_array(graph, dtype=float)
    deg = A.sum(axis=1)

    P = np.zeros_like(A, dtype=float)
    for i in range(A.shape[0]):
        if deg[i] > 0:
            P[i] = A[i] / deg[i]
        else:
            P[i, i] = 1.0

    return P

In [5]:
def normalized_adj_matrix_labeled(graph):
    nodes = list(graph.nodes())
    n = len(nodes)
    idx = {u: i for i, u in enumerate(nodes)}
    A_labels = {}
    deg = np.zeros(n, dtype=float)

    for u, v, data in graph.edges(data=True):
        i, j = idx[u], idx[v]
        lab = int(data["label"])
        if lab not in A_labels:
            A_labels[lab] = np.zeros((n, n), dtype=float)
        A_labels[lab][i, j] = 1.0
        A_labels[lab][j, i] = 1.0
        deg[i] += 1.0
        deg[j] += 1.0

    P_labels = {}
    for lab, A in A_labels.items():
        P = np.zeros_like(A)
        for i in range(n):
            if deg[i] > 0:
                P[i, :] = A[i, :] / deg[i]
        P_labels[lab] = P

    return P_labels

In [6]:
def uniform_dist(n):
    return np.ones(n, dtype=float) / n

def random_dist(n):
    x = np.random.random(n)
    return x / x.sum()

In [7]:
def mu_func_gen(kind="exp", lmbd=0.1):
    if kind == "exp":
        def mu(k):
            return (lmbd ** k) / math.factorial(k)
        return mu
    if kind == "geom":
        def mu(k):
            return lmbd ** k
        return mu
    raise ValueError(f"unknown kind: {kind}")

## Brute Force computations

#### Unlabeled

In [8]:
def random_walk_kernel(P1, P2, v1, v2, w1, w2, mu_func, kernel_type="general", max_iter=30):
    n1, n2 = len(v1), len(v2)
    W = np.kron(P1, P2)
    v = np.kron(v1, v2)
    w = np.kron(w1, w2)
    
    if kernel_type == "exp":
        lmbd = mu_func(1)
        S = la.expm(lmbd * W)
        
    elif kernel_type == "geom":
        lmbd = mu_func(1)
        I = np.eye(W.shape[0], dtype=float)
        S = np.linalg.inv(I - lmbd * W)
        
    else:
        Wk = np.eye(W.shape[0], dtype=float)
        S = mu_func(0) * Wk
        for k in range(1, max_iter + 1):
            Wk = Wk @ W
            S += mu_func(k) * Wk
            
    return float(v @ (S @ w))

#### Labeled

In [9]:
def random_walk_kernel_labeled(P1_labeled, P2_labeled, v1, v2, w1, w2, mu_func, kernel_type="general", max_iter=30):
    n1, n2 = len(v1), len(v2)
    W = np.zeros((n1 * n2, n1 * n2), dtype=float)
    v = np.kron(v1, v2)
    w = np.kron(w1, w2)

    for label in set(P1_labeled.keys()) & set(P2_labeled.keys()):
        W += np.kron(P1_labeled[label], P2_labeled[label])
        
    if kernel_type == "exp":
        lmbd = mu_func(1)
        S = la.expm(lmbd * W)
        
    elif kernel_type == "geom":
        lmbd = mu_func(1)
        I = np.eye(W.shape[0], dtype=float)
        S = np.linalg.inv(I - lmbd * W)
        
    else:
        Wk = np.eye(W.shape[0], dtype=float)
        S = mu_func(0) * Wk
        for k in range(1, max_iter + 1):
            Wk = Wk @ W
            S += mu_func(k) * Wk
            
    return float(v @ (S @ w))

## Existing Optimizations

Sylvester Equation, Fixed Point and Conjugate Gradient methods are from Vishwanathan et al. 2010 are applied only for geometric kernel

### Sylvester Equation 

#### Unlabeled

In [10]:
def random_walk_kernel_sylvester(P1, P2, v1, v2, w1, w2, mu_func):
    """
    geometric random-walk kernel via Schur-based Sylvester equation.
    """
    lmbd = mu_func(1)
    W0 = np.outer(w2, w1)
    V0 = np.outer(v2, v1)
    T2, U2 = la.schur(P2, output="complex")
    T1, U1 = la.schur(P1.T, output="complex")
    C = U2.conj().T @ W0 @ U1
    n2, n1 = C.shape
    Y = np.zeros((n2, n1), dtype=complex)

    for j in range(n1):
        rhs = C[:, j].copy()
        if j > 0:
            accum = np.zeros(n2, dtype=complex)
            for k in range(j):
                accum += Y[:, k] * T1[k, j]
            rhs += lmbd * (T2 @ accum)
        A = np.eye(n2, dtype=complex) - lmbd * T1[j, j] * T2
        Y[:, j] = np.linalg.solve(A, rhs)

    M = U2 @ Y @ U1.conj().T
    val = np.sum(V0 * M)

    return float(np.real_if_close(val))

#### Labeled

### Fixed Point Iteration

#### Unlabeled

In [11]:
def random_walk_kernel_fixed_point(P1, P2, v1, v2, w1, w2, mu_func, eps=1e-30, max_iter=1000):
    n1, n2 = P1.shape[0], P2.shape[0]
    lmbd = mu_func(1)
    w0 = np.outer(w2, w1) # vec(w0) = w1 \otimes w2
    v0 = np.outer(v2, v1)
    x = w0.copy()

    for i in range(max_iter):
        x_new = w0 + lmbd * (P2 @ x @ P1.T)
        if np.linalg.norm(x_new - x, ord="fro") <= eps:
            x = x_new
            break
        x = x_new
        
    return float(np.sum(v0 * x))

#### Labeled

In [12]:
def random_walk_kernel_fixed_point_labeled(P1_labeled, P2_labeled, v1, v2, w1, w2, mu_func, eps=1e-30, max_iter=1000):
    common_labels = set(P1_labeled.keys()) & set(P2_labeled.keys())
    w0 = np.outer(w2, w1)
    v0 = np.outer(v2, v1)
    x = w0.copy()
    lmbd = mu_func(1)

    for i in range(max_iter):
        x_new = w0.copy()
        for label in common_labels:
            x_new += lmbd * (P2_labeled[label] @ x @ P1_labeled[label].T)
        
        if np.linalg.norm(x_new - x, ord="fro") <= eps:
            x = x_new
            break
        
        x = x_new

    return float(np.sum(v0 * x))

### Conjugate gradient

#### Unlabeled

In [13]:
def random_walk_kernel_cg(P1, P2, v1, v2, w1, w2, mu_func, eps=1e-30, max_iter=1000):
    n1, n2 = P1.shape[0], P2.shape[0]
    v = np.kron(v1, v2)
    w = np.kron(w1, w2)
    lmbd = mu_func(1)
    
    def matvec(x):
        X = x.reshape((n2, n1), order="F")
        Y = X - lmbd * (P2 @ X @ P1.T)
        return Y.reshape(-1, order="F")

    A = LinearOperator(shape=(n1 * n2, n1 * n2),matvec=matvec, dtype=float)
    x, info = cg(A, w, rtol=eps, maxiter=max_iter)
    return float(v @ x)

#### Labeled

In [14]:
def random_walk_kernel_cg_labeled(P1_labeled, P2_labeled, v1, v2, w1, w2, mu_func, tol=1e-30, max_iter=1000):
    common_labels = set(P1_labeled.keys()) & set(P2_labeled.keys())
    n1, n2 = len(v1),len(v2)
    v = np.kron(v1, v2)
    w = np.kron(w1, w2)
    lmbd = mu_func(1)

    def matvec(x):
        X = x.reshape((n2, n1), order="F")
        Y = X.copy()
        for label in common_labels:
            Y -= lmbd * (P1_labeled[label] @ X @ P2_labeled[label].T)
        return Y.reshape(-1, order="F")

    operator = LinearOperator(shape=(n1 * n2, n1 * n2), matvec=matvec, dtype=float)
    x, info = cg(operator, w, rtol=tol, maxiter=max_iter)
    return float(v @ x)

#### Unlabeled test

In [15]:
n = 50
G1 = graph_generator(n, kind="er", seed=1)
G2 = graph_generator(n, kind="er", seed=2)

P1 = normalized_adj_matrix(G1)
P2 = normalized_adj_matrix(G2)

v1 = uniform_dist(n)
v2 = uniform_dist(n)
w1 = uniform_dist(n)
w2 = uniform_dist(n)

mu_exp = mu_func_gen("exp", lmbd=0.1)
mu_geom = mu_func_gen("geom", lmbd=0.1)

#exp
k_exp_exact = random_walk_kernel(P1, P2, v1, v2, w1, w2, mu_exp, kernel_type="exp")
k_exp_trunc = random_walk_kernel(P1, P2, v1, v2, w1, w2, mu_exp, kernel_type="general", max_iter=20)
print("exp exact :", k_exp_exact)
print("exp trunc :", k_exp_trunc)

#geom
k_geom_exact = random_walk_kernel(P1, P2, v1, v2, w1, w2, mu_geom, kernel_type="geom")
k_geom_trunc = random_walk_kernel(P1, P2, v1, v2, w1, w2, mu_geom, kernel_type="general", max_iter=20)

k_geom_sylv = random_walk_kernel_sylvester(P1, P2, v1, v2, w1, w2, mu_geom)
k_geom_fp = random_walk_kernel_fixed_point(P1, P2, v1, v2, w1, w2, mu_geom)
k_geom_cg = random_walk_kernel_cg(P1, P2, v1, v2, w1, w2, mu_geom)

print("geom exact:", k_geom_exact)
print("geom trunc:", k_geom_trunc)
print("sylvester:", k_geom_sylv)
print("fixed point:", k_geom_fp)
print("cg:", k_geom_cg)

exp exact : 0.00044206836723025835
exp trunc : 0.00044206836723025835
geom exact: 0.00044444444444444555
geom trunc: 0.00044444444444444555
sylvester: 0.00044444444444444577
fixed point: 0.00044444444444444447
cg: 0.00044444444444444544


#### Labeled test

In [16]:
# Labeled
n = 50
n_labels = 3
G1 = graph_generator_labeled(n, kind="er", n_labels=n_labels, seed=1)
G2 = graph_generator_labeled(n, kind="er", n_labels=n_labels, seed=2)

P1_labeled = normalized_adj_matrix_labeled(G1)
P2_labeled = normalized_adj_matrix_labeled(G2)

v1 = uniform_dist(n)
v2 = uniform_dist(n)
w1 = uniform_dist(n)
w2 = uniform_dist(n)

mu_exp = mu_func_gen("exp", lmbd=0.1)
mu_geom = mu_func_gen("geom", lmbd=0.1)

k_exp_exact = random_walk_kernel_labeled(P1_labeled, P2_labeled, v1, v2, w1, w2, mu_exp, kernel_type="exp")
k_exp_trunc = random_walk_kernel_labeled(P1_labeled, P2_labeled, v1, v2, w1, w2, mu_exp, kernel_type="general", max_iter=20)
print("exp exact :", k_exp_exact)
print("exp trunc :", k_exp_trunc)

k_geom_exact = random_walk_kernel_labeled(P1_labeled, P2_labeled, v1, v2, w1, w2, mu_geom, kernel_type="geom")
k_geom_trunc = random_walk_kernel_labeled(P1_labeled, P2_labeled, v1, v2, w1, w2, mu_geom, kernel_type="general", max_iter=20)
# k_geom_sylv = random_walk_kernel_sylvester_labeled(P1_labeled, P2_labeled, v1, v2, w1, w2, mu_geom)
k_geom_fixed_point = random_walk_kernel_fixed_point_labeled(P1_labeled, P2_labeled, v1, v2, w1, w2, mu_geom)
k_geom_cg = random_walk_kernel_cg_labeled(P1_labeled, P2_labeled, v1, v2, w1, w2, mu_geom)

print("geom exact:", k_geom_exact)
print("geom trunc:", k_geom_trunc)
print("fixed point:", k_geom_fixed_point)
print("cg:", k_geom_cg)

exp exact : 0.00041025820688906105
exp trunc : 0.00041025820688906105
geom exact: 0.00041050786806759795
geom trunc: 0.00041050786806759795
fixed point: 0.00041050786806759844
cg: 0.00041050786806759795


### Graph Voyagers 

from Choromanski et al. 2025. Changes were made to tailor it to make algorithm more correctly and tailor it for geometric kernel

In [17]:
SIGMA = 0.1
LAMBDA_COEFF = 0.1
P_HALT = 0.2
NB_RANDOM_WALKS = 1000
RANDOM_SEED = 180
BIG_NUMBER = 2000

np.random.seed(RANDOM_SEED)
t_variables = np.random.uniform(size=(2 * BIG_NUMBER, 2 * BIG_NUMBER))
g_variables = np.where(
    np.random.normal(size=(2 * BIG_NUMBER, 2 * BIG_NUMBER)) > 0.0,
    1.0,
    -1.0,
)

def adj_matrix_to_lists(P):
    """
    Convert row-stochastic transition matrix into adjacency + weight lists.
    """
    adj_lists = []
    weight_lists = []

    for i in range(P.shape[0]):
        neigh = np.where(P[i] > 0.0)[0].tolist()
        weights = P[i, neigh].tolist()
        adj_lists.append(neigh)
        weight_lists.append(weights)

    return adj_lists, weight_lists


def f_func_diffusion(i, lambda_coeff):
    """
    Same modulation as in GVoy's code.
    """
    return lambda_coeff ** i / (2 ** i * scipy.special.factorial(i))


def create_pq_vectors(
    adj_lists,
    weight_lists,
    anchor_points_dict,
    p_halt,
    nb_random_walks,
    f,
    is_left,
    base_nb_walk_index,
):
    """
    Same algorithm, only adapted to notebook naming/style.
    """
    n = len(adj_lists)
    s_matrix = np.zeros((nb_random_walks, len(anchor_points_dict), n))

    for w in range(nb_random_walks):
        for k in range(n):
            load = 1.0
            step_counter = 0
            current_vertex = k

            x_index = is_left * BIG_NUMBER + step_counter
            y_index = is_left * BIG_NUMBER + w + base_nb_walk_index

            if current_vertex in anchor_points_dict:
                add_term = load * np.sqrt(f(step_counter))
                add_term *= g_variables[x_index][y_index]
                s_matrix[w, anchor_points_dict[current_vertex], k] += add_term

            if adj_lists[current_vertex] == []:
                continue

            while t_variables[x_index][y_index] > p_halt:
                if step_counter >= BIG_NUMBER - 1:
                    break

                rnd_index = int(rnd.uniform(0, 1) * len(adj_lists[current_vertex]))
                p_uv = weight_lists[current_vertex][rnd_index]

                load *= p_uv
                load *= 1.0 / np.sqrt(1.0 - p_halt)

                step_counter += 1
                current_vertex = adj_lists[current_vertex][rnd_index]

                x_index = is_left * BIG_NUMBER + step_counter
                y_index = is_left * BIG_NUMBER + w + base_nb_walk_index

                if current_vertex in anchor_points_dict:
                    add_term = load * np.sqrt(f(step_counter))
                    add_term *= g_variables[x_index][y_index]
                    s_matrix[w, anchor_points_dict[current_vertex], k] += add_term

                if adj_lists[current_vertex] == []:
                    break

    return s_matrix


def approximate_graph_kernel_value(
    P1,
    P2,
    v1,
    v2,
    w1,
    w2,
    anchor_fraction=1.0,
    base_nb_walk_index=0,
    kernel_type="exponential",
    lambda_coeff=LAMBDA_COEFF,
    p_halt=P_HALT,
    nb_random_walks=NB_RANDOM_WALKS,
):
    P1_adj_lists, P1_weight_lists = adj_matrix_to_lists(P1)
    P2_adj_lists, P2_weight_lists = adj_matrix_to_lists(P2)

    n1 = len(P1_adj_lists)
    n2 = len(P2_adj_lists)

    nb_anc1 = max(1, int(anchor_fraction * n1))
    nb_anc2 = max(1, int(anchor_fraction * n2))

    anc1 = np.random.choice(np.arange(n1), size=nb_anc1, replace=False)
    anc2 = np.random.choice(np.arange(n2), size=nb_anc2, replace=False)
    anc1 = np.sort(anc1)
    anc2 = np.sort(anc2)

    anc1_dict = dict(zip(anc1, np.arange(nb_anc1)))
    anc2_dict = dict(zip(anc2, np.arange(nb_anc2)))

    if kernel_type != "exponential":
        raise ValueError("This implementation keeps only the exponential/diffusion case.")

    f_function = lambda i: f_func_diffusion(i, lambda_coeff)

    p1 = create_pq_vectors(
        P1_adj_lists, P1_weight_lists, anc1_dict,
        p_halt, nb_random_walks, f_function, 0, base_nb_walk_index
    )
    p2 = create_pq_vectors(
        P2_adj_lists, P2_weight_lists, anc2_dict,
        p_halt, nb_random_walks, f_function, 0, base_nb_walk_index
    )
    q1 = create_pq_vectors(
        P1_adj_lists, P1_weight_lists, anc1_dict,
        p_halt, nb_random_walks, f_function, 1, base_nb_walk_index
    )
    q2 = create_pq_vectors(
        P2_adj_lists, P2_weight_lists, anc2_dict,
        p_halt, nb_random_walks, f_function, 1, base_nb_walk_index
    )

    P1_lat = np.einsum(
        "br,br->br",
        np.einsum("brN,N->br", p1, v1),
        np.einsum("brN,N->br", q1, w1),
    )
    P2_lat = np.einsum(
        "br,br->br",
        np.einsum("brN,N->br", p2, v2),
        np.einsum("brN,N->br", q2, w2),
    )

    final_batch = np.einsum("bx,by->bxy", P1_lat, P2_lat)
    return (1.0 / nb_random_walks) * np.sum(final_batch)


def approximate_graph_kernel_value_with_blocks(
    P1,
    P2,
    v1,
    v2,
    w1,
    w2,
    anchor_fraction=1.0,
    kernel_type="exponential",
    lambda_coeff=LAMBDA_COEFF,
    p_halt=P_HALT,
    nb_random_walks=NB_RANDOM_WALKS,
    block_size=NB_RANDOM_WALKS,
):
    if nb_random_walks % block_size != 0:
        raise ValueError("nb_random_walks must be divisible by block_size.")

    approx_val = 0.0
    for i in range(nb_random_walks // block_size):
        approx_val += approximate_graph_kernel_value(
            P1, P2, v1, v2, w1, w2,
            anchor_fraction=anchor_fraction,
            base_nb_walk_index=i * block_size,
            kernel_type=kernel_type,
            lambda_coeff=lambda_coeff,
            p_halt=p_halt,
            nb_random_walks=block_size,
        )

    return approx_val * (block_size / nb_random_walks)

In [ ]:
N = 50
lambda_coeff = 0.01
nb_random_walks = 1000

G1 = graph_generator(N, kind="er", seed=1)
G2 = graph_generator(N, kind="er", seed=2)

P1 = normalized_adj_matrix(G1)
P2 = normalized_adj_matrix(G2)

v1 = uniform_dist(N)
v2 = uniform_dist(N)
w1 = uniform_dist(N)
w2 = uniform_dist(N)

mu_exp = mu_func_gen("exp", lmbd=lambda_coeff)

t0 = time.perf_counter()
exact = random_walk_kernel(
    P1, P2, v1, v2, w1, w2,
    mu_func=mu_exp,
    kernel_type="exp",
)
t1 = time.perf_counter()

np.random.seed(RANDOM_SEED)

t2 = time.perf_counter()
approx = approximate_graph_kernel_value_with_blocks(
    P1, P2, v1, v2, w1, w2,
    anchor_fraction=1.0,
    kernel_type="exponential",
    lambda_coeff=lambda_coeff,
    p_halt=P_HALT,
    nb_random_walks=nb_random_walks,
    block_size=nb_random_walks,
)
t3 = time.perf_counter()

abs_err = float(np.abs(exact - approx))
rel_err = float(abs_err / (np.abs(exact)))

print("GVoys")
print(f"n={N}, lmbd={lambda_coeff}, n_walks={nb_random_walks}, p_halt={P_HALT}")
print(f"exact: {exact}  time={t1 - t0}s")
print(f"gvoys: {approx}  time={t3 - t2}s")
print(f"abs err: {abs_err}")
print(f"rel err: {rel_err}")

GVoys
n=50, lmbd=0.01, n_walks=1000, p_halt=0.2
exact: 0.0004040200668336681  time=3.2314272090006853s
approx: 0.0004041596214576056  time=10.521373832998506s
abs err: 1.395546239375085e-07
rel err: 0.00034541508057064414


In [19]:
SIGMA = 0.1
LAMBDA_COEFF = 0.1
P_HALT = 0.2
NB_RANDOM_WALKS = 1000
RANDOM_SEED = 180
BIG_NUMBER = 2000

np.random.seed(RANDOM_SEED)
t_variables = np.random.uniform(size=(2 * BIG_NUMBER, 2 * BIG_NUMBER))
g_variables = np.where(
    np.random.normal(size=(2 * BIG_NUMBER, 2 * BIG_NUMBER)) > 0.0,
    1.0,
    -1.0,
)

def adj_matrix_to_lists(P):
    adj_lists = []
    weight_lists = []
    for i in range(P.shape[0]):
        neigh = np.where(P[i] > 0.0)[0].tolist()
        weights = P[i, neigh].tolist()
        adj_lists.append(neigh)
        weight_lists.append(weights)

    return adj_lists, weight_lists


def f_func_diffusion(i, lambda_coeff):
    # exponential kernel modulation
    return lambda_coeff ** i / (2 ** i * scipy.special.factorial(i))

def f_func_geometric(i, lambda_coeff):
    # geometric kernel modulation
    return lambda_coeff ** i


def create_pq_vectors(
    adj_lists,
    weight_lists,
    anchor_points_dict,
    p_halt,
    nb_random_walks,
    f,
    is_left,
    base_nb_walk_index,
):
    n = len(adj_lists)
    s_matrix = np.zeros((nb_random_walks, len(anchor_points_dict), n))

    for w in range(nb_random_walks):
        for k in range(n):
            load = 1.0
            step_counter = 0
            current_vertex = k

            x_index = is_left * BIG_NUMBER + step_counter
            y_index = is_left * BIG_NUMBER + w + base_nb_walk_index

            if current_vertex in anchor_points_dict:
                add_term = load * np.sqrt(f(step_counter))
                add_term *= g_variables[x_index][y_index]
                s_matrix[w, anchor_points_dict[current_vertex], k] += add_term

            if adj_lists[current_vertex] == []:
                continue

            while t_variables[x_index][y_index] > p_halt:
                if step_counter >= BIG_NUMBER - 1:
                    break

                rnd_index = int(rnd.uniform(0, 1) * len(adj_lists[current_vertex]))
                p_uv = weight_lists[current_vertex][rnd_index]

                load *= p_uv
                load *= 1.0 / np.sqrt(1.0 - p_halt)

                step_counter += 1
                current_vertex = adj_lists[current_vertex][rnd_index]

                x_index = is_left * BIG_NUMBER + step_counter
                y_index = is_left * BIG_NUMBER + w + base_nb_walk_index

                if current_vertex in anchor_points_dict:
                    add_term = load * np.sqrt(f(step_counter))
                    add_term *= g_variables[x_index][y_index]
                    s_matrix[w, anchor_points_dict[current_vertex], k] += add_term

                if adj_lists[current_vertex] == []:
                    break

    return s_matrix


def approximate_graph_kernel_value(
    P1,
    P2,
    v1,
    v2,
    w1,
    w2,
    anchor_fraction=1.0,
    base_nb_walk_index=0,
    kernel_type="exponential",
    lambda_coeff=LAMBDA_COEFF,
    p_halt=P_HALT,
    nb_random_walks=NB_RANDOM_WALKS,
):
    P1_adj_lists, P1_weight_lists = adj_matrix_to_lists(P1)
    P2_adj_lists, P2_weight_lists = adj_matrix_to_lists(P2)

    n1 = len(P1_adj_lists)
    n2 = len(P2_adj_lists)

    nb_anc1 = max(1, int(anchor_fraction * n1))
    nb_anc2 = max(1, int(anchor_fraction * n2))

    anc1 = np.random.choice(np.arange(n1), size=nb_anc1, replace=False)
    anc2 = np.random.choice(np.arange(n2), size=nb_anc2, replace=False)
    anc1 = np.sort(anc1)
    anc2 = np.sort(anc2)

    anc1_dict = dict(zip(anc1, np.arange(nb_anc1)))
    anc2_dict = dict(zip(anc2, np.arange(nb_anc2)))

    if kernel_type == "exponential":
        f_function = lambda i: f_func_diffusion(i, lambda_coeff)
    elif kernel_type == "geometric":
        f_function = lambda i: f_func_geometric(i, lambda_coeff)
    else:
        raise ValueError("kernel_type must be 'exponential' or 'geometric'")

    p1 = create_pq_vectors(
        P1_adj_lists, P1_weight_lists, anc1_dict,
        p_halt, nb_random_walks, f_function, 0, base_nb_walk_index
    )
    p2 = create_pq_vectors(
        P2_adj_lists, P2_weight_lists, anc2_dict,
        p_halt, nb_random_walks, f_function, 0, base_nb_walk_index
    )
    q1 = create_pq_vectors(
        P1_adj_lists, P1_weight_lists, anc1_dict,
        p_halt, nb_random_walks, f_function, 1, base_nb_walk_index
    )
    q2 = create_pq_vectors(
        P2_adj_lists, P2_weight_lists, anc2_dict,
        p_halt, nb_random_walks, f_function, 1, base_nb_walk_index
    )

    P1_lat = np.einsum(
        "br,br->br",
        np.einsum("brN,N->br", p1, v1),
        np.einsum("brN,N->br", q1, w1),
    )
    P2_lat = np.einsum(
        "br,br->br",
        np.einsum("brN,N->br", p2, v2),
        np.einsum("brN,N->br", q2, w2),
    )

    final_batch = np.einsum("bx,by->bxy", P1_lat, P2_lat)
    return (1.0 / nb_random_walks) * np.sum(final_batch)


def approximate_graph_kernel_value_with_blocks(
    P1,
    P2,
    v1,
    v2,
    w1,
    w2,
    anchor_fraction=1.0,
    kernel_type="exponential",
    lambda_coeff=LAMBDA_COEFF,
    p_halt=P_HALT,
    nb_random_walks=NB_RANDOM_WALKS,
    block_size=NB_RANDOM_WALKS,
):
    if nb_random_walks % block_size != 0:
        raise ValueError("nb_random_walks must be divisible by block_size.")

    approx_val = 0.0
    for i in range(nb_random_walks // block_size):
        approx_val += approximate_graph_kernel_value(
            P1, P2, v1, v2, w1, w2,
            anchor_fraction=anchor_fraction,
            base_nb_walk_index=i * block_size,
            kernel_type=kernel_type,
            lambda_coeff=lambda_coeff,
            p_halt=p_halt,
            nb_random_walks=block_size,
        )

    return approx_val * (block_size / nb_random_walks)

In [ ]:
N = 50
lmbd = 0.01
nb_random_walks = 1000

G1 = graph_generator(N, kind="er", seed=1)
G2 = graph_generator(N, kind="er", seed=2)

P1 = normalized_adj_matrix(G1)
P2 = normalized_adj_matrix(G2)

v1 = uniform_dist(N)
v2 = uniform_dist(N)
w1 = uniform_dist(N)
w2 = uniform_dist(N)

mu_geom = mu_func_gen("geom", lmbd=lmbd)

t0 = time.perf_counter()
exact_geom = random_walk_kernel(P1, P2, v1, v2, w1, w2, mu_func=mu_geom, kernel_type="geom")
t1 = time.perf_counter()

np.random.seed(RANDOM_SEED)
t2 = time.perf_counter()
approx_geom = approximate_graph_kernel_value_with_blocks(
    P1, P2, v1, v2, w1, w2,
    anchor_fraction=1.0,
    kernel_type="geometric",
    lambda_coeff=lmbd,
    p_halt=P_HALT,
    nb_random_walks=nb_random_walks,
    block_size=nb_random_walks,
)
t3 = time.perf_counter()

print("geom kernel")
print(f"exact  : {exact_geom:.10g}  time={t1-t0:.3f}s")
print(f"gvoys : {approx_geom:.10g}  time={t3-t2:.3f}s")
print(f"abs err: {abs(exact_geom - approx_geom):.3g}")
print(f"rel err: {abs(exact_geom - approx_geom)/(abs(exact_geom)):.3g}")

=== Geometric kernel ===
Exact  : 0.000404040404  time=0.838s
Approx : 0.0004067880182  time=2.154s
Abs err: 2.75e-06
Rel err: 0.0068


## Monte Carlo Random Walk Kernel

### Unlabeled

In [21]:
def kernel_normalizer(kind, mu_func):
    if kind == "exp":
        lmbd = mu_func(1)
        return math.exp(lmbd)
    if kind == "geom":
        lmbd = mu_func(1)
        return 1.0 / (1.0 - lmbd)
    raise ValueError(f"unsupported kind: {kind}")

def sample_length(kind, mu_func, rng):
    if kind == "exp":
        lmbd = mu_func(1)
        return rng.poisson(lmbd)
    if kind == "geom":
        lmbd = mu_func(1)
        return rng.geometric(1.0 - lmbd) - 1
    raise ValueError(f"unsupported kind: {kind}")
    
def get_samples(P, v, w, shared_random_variables, n_samples, rng):
    samples = np.zeros(n_samples, dtype=float)
    for i in range(n_samples):
        len_walk = shared_random_variables[i]
        x = rng.choice(len(v), p=v) #sample starting point from distribution v
        for j in range(len_walk):
            x = rng.choice(P.shape[1], p=P[x]) #random walk step
        samples[i] = w[x]
    return samples

def random_walk_kernel_mc(P1, P2, v1, v2, w1, w2, mu_func, kind, n_samples=100, seed=42):
    rng = np.random.default_rng(seed)
    C = kernel_normalizer(kind, mu_func)
    shared_random_variables = np.zeros(n_samples, dtype=int)
    for i in range(n_samples):
        shared_random_variables[i] = sample_length(kind, mu_func, rng)  
    g1_samples = get_samples(P1, v1, w1, shared_random_variables, n_samples, rng)
    g2_samples = get_samples(P2, v2, w2, shared_random_variables, n_samples, rng)
    return C * (g1_samples * g2_samples).mean()
    

In [ ]:
N = 50
lmbd = 0.01

G1 = graph_generator(N, kind="er", seed=1)
G2 = graph_generator(N, kind="er", seed=2)

P1 = normalized_adj_matrix(G1)
P2 = normalized_adj_matrix(G2)

v1 = uniform_dist(N)
v2 = uniform_dist(N)
w1 = uniform_dist(N)
w2 = uniform_dist(N)

#exp test
mu_exp = mu_func_gen("exp", lmbd=lmbd)
exact_exp = random_walk_kernel(P1, P2, v1, v2, w1, w2, mu_func=mu_exp, kernel_type="exp")
approx_exp = random_walk_kernel_mc(P1, P2, v1, v2, w1, w2, mu_func=mu_exp, kind="exp", n_samples=1000, seed=42)
print("exact exp :", exact_exp)
print("mc exp:", approx_exp)
print("abs err:", abs(exact_exp - approx_exp))
print(f"rel err: {abs(exact_exp - approx_exp)/(abs(exact_exp))}")

#geom test
mu_geom = mu_func_gen("geom", lmbd=lmbd)
exact_geom = random_walk_kernel(P1, P2, v1, v2, w1, w2, mu_func=mu_geom, kernel_type="geom")
approx_geom = random_walk_kernel_mc(P1, P2, v1, v2, w1, w2, mu_func=mu_geom, kind="geom", n_samples=1000, seed=42)
print("exact geom :", exact_geom)
print("mc geom:", approx_geom)
print("abs err:", abs(exact_geom - approx_geom))
print(f"rel err: {abs(exact_geom - approx_geom)/(abs(exact_geom))}")

exact exp : 0.0004040200668336681
approx exp: 0.0004040200668336673
abs err: 8.131516293641283e-19
rel err: 2.0126515886620467e-15
exact geom : 0.000404040404040405
approx geom: 0.0004040404040404042
abs err: 8.131516293641283e-19
rel err: 2.012550282676213e-15


linear time with respect to dataset size

In [23]:
def random_walk_kernel_mc_dataset(Ps, vs, ws, mu_func, kind, n_samples, seed):
    rng = np.random.default_rng(seed)
    C = kernel_normalizer(kind, mu_func)
    shared_random_variables = np.zeros(n_samples, dtype=int)
    for i in range(n_samples):
        shared_random_variables[i] = sample_length(kind, mu_func, rng)
     
    n_graphs = len(Ps)
    graph_samples = np.zeros(n_graphs)
    for i in range(len(Ps)):
        graph_samples[i] = get_samples(Ps[i], vs[i], ws[i], shared_random_variables, n_samples)
        
    return graph_samples, C

### Labeled

## Benchmarks

### Unlabeled

In [24]:
def bench(dataset, name, mu_func, n_graphs, n_samples_mc, n_samples_gvoys, seed=42):
    ...

### Labeled